In [1]:
from pathlib import Path

In [2]:
from atlas.io import get_valid_slice_folders

In [3]:
#testing data path
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS\data\zeinab")
series_folder = Path(r"/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd")

In [4]:
series_list, _ = get_valid_slice_folders(series_folder)

In [5]:
print(f"\nFound {len(series_list)} series.")

for f in series_list:
    print(f)


Found 20 series.
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_009_183996284
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_003_1082014046
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_004_520641757
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_007_219502197
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_006_1441661872
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_005_1107052629
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_020_876249941
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_010_1018560366
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_008_564872779
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_017_314689960
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_012_343837657
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_019_971884407
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_014_2134637132
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_021_2072832173
/home/xgupke/Documents/data/ZEINAB-ROI

In [6]:
import json

import numpy as np
import tifffile as tiff
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

from atlas.image_analysis import calculate_mask_roi, mask_low_and_saturation

from atlas.io import extract_s_number
from atlas.stitching import (
    stitch_ATLAS_tiles,
    add_tile_overlap_columns,
    apply_transforms_and_stitch,
    build_adjacency_matrix_from_costs,
    build_transform_dict_from_mst,
    get_tiles_dataframe,
    match_tiles,
)

In [7]:
buffer_in_microns = 1

max_shift_in_pixles = 200

# iterate over all folders in the series
for raw_data_folder in series_list:
    # Iterate over all items in the folder
    for file in raw_data_folder.iterdir():  
        # Check if it's a file with the desired extension
        if file.is_file() and file.suffix == ".ve-mif":
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            # use naming convention to find the first tif
            first_tif_path = next(raw_data_folder.glob("*.tif"))
            print(first_tif_path)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")

                try:
                    stitched_img, mif_tile_df, transform_dict = stitch_ATLAS_tiles(
                        mif_file,
                        buffer_microns=buffer_in_microns,
                        max_shift_pixels=max_shift_in_pixles,
                    )

                    # Save the stitched image as a TIFF file
                    tiff.imwrite(output_tif_path, np.flipud(stitched_img))
                    

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            

File with '.ve-mif' extension found: MosaicInfo_S_009_183996284.ve-mif
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_009_183996284/Tile_r3-c1_S_009_183996284.tif
✅ Skipping: stitched_image_S_9.tiff already exists.
File with '.ve-mif' extension found: MosaicInfo_S_003_1082014046.ve-mif
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_003_1082014046/Tile_r1-c1_S_003_1082014046.tif
✅ Skipping: stitched_image_S_3.tiff already exists.
File with '.ve-mif' extension found: MosaicInfo_S_004_520641757.ve-mif
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_004_520641757/Tile_r2-c1_S_004_520641757.tif
✅ Skipping: stitched_image_S_4.tiff already exists.
File with '.ve-mif' extension found: MosaicInfo_S_007_219502197.ve-mif
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_007_219502197/Tile_r2-c1_S_007_219502197.tif
✅ Skipping: stitched_image_S_7.tiff already exists.
File with '.ve-mif' extension found: MosaicInfo_S_006_1441661872.ve-mif
/home/xgupke/Documents/data/ZEINAB-ROI-w1